# pegasus text summarization fine-tuning on samsum datset

this notebook contains the complete, self-contained code to fine-tune teh **google pegasus** model (`google/pegasus-cnn_dailymail`) on teh **samsum dialoge datset** using a gpu on google colab.

### datset source:
we download the samsum datset directly from teh github repository zip file:
`https://github.com/entbappy/branching-tutorial/raw/master/summarizer-data.zip`

this ensures we use the exact same dataset splits that are in your local project.

## 1. install required libaries

colab enviornments need these libaries preinstalled. run teh cell below to set them up.

In [ ]:
!pip install transformers[sentencepiece] datasets evaluate rouge_score accelerate py7zr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 17.1 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5a25e1c4ce3c69b7fe04750d569f6d446299a97b95d620c10a3b732d585f3b8d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


## 2. download adn extract teh samsum datset from github

we download the zip file from github directly to teh colab enviornment adn extract it.

In [ ]:
import urllib.request as request
import zipfile
import os

url = "https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip"
zip_path = "summarizer-data.zip"


if os.path.exists(zip_path):
    os.remove(zip_path)

print("Downloading dataset...")
request.urlretrieve(url, zip_path)
print("Download completed!")


print("\nContents of the ZIP:")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    for file in zip_ref.namelist():
        print(file)

    print("\nExtracting files...")
    zip_ref.extractall(".")

print("\nExtraction completed!")


print("\nCurrent directory contents:")
print(os.listdir("."))

print("\nExtracted file structure:")
for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}    {f}")

Download completed!

Contents of the ZIP:
samsum-test.csv
samsum-train.csv
samsum-validation.csv
samsum_dataset/
samsum_dataset/dataset_dict.json
samsum_dataset/test/
samsum_dataset/test/data-00000-of-00001.arrow
samsum_dataset/test/dataset_info.json
samsum_dataset/test/state.json
samsum_dataset/train/
samsum_dataset/train/data-00000-of-00001.arrow
samsum_dataset/train/dataset_info.json
samsum_dataset/train/state.json
samsum_dataset/validation/
samsum_dataset/validation/data-00000-of-00001.arrow
samsum_dataset/validation/dataset_info.json
samsum_dataset/validation/state.json

Extracting files...

Extraction completed!

Current directory contents:
['.config', 'samsum-train.csv', 'samsum-validation.csv', 'samsum_dataset', 'samsum-test.csv', 'summarizer-data.zip', 'sample_data']

Extracted file structure:
./
    samsum-train.csv
    samsum-validation.csv
    samsum-test.csv
    summarizer-data.zip
    .config/
        default_configs.db
        hidden_gcloud_config_universe_descriptor_dat

## 3. load tokenizer & preprocess data

we load the extracted hugging face arrow datset directory, load teh base pegasus toknizer, and map our inputs into tokenized features.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_from_disk
import os


model_ckpt = "google/pegasus-cnn_dailymail"
print(f"Loading tokenizer: {model_ckpt}...")
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


data_path = "samsum_dataset"

if os.path.exists(data_path):
    dataset_samsum = load_from_disk(data_path)
    print("Loaded dataset successfully:")
    print(dataset_samsum)
else:
    raise FileNotFoundError(f"Could not find dataset directory at: {data_path}")

# Tokenization function
def convert_examples_to_features(example_batch):
    # Tokenize dialogues
    model_inputs = tokenizer(
        example_batch["dialogue"],
        max_length=1024,
        truncation=True
    )

    # Tokenize summaries
    labels = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization to all dataset splits
print("Tokenizing dataset splits...")
dataset_samsum_pt = dataset_samsum.map(
    convert_examples_to_features,
    batched=True
)

print("Tokenization completed successfully!")
print("Available splits:", list(dataset_samsum_pt.keys()))

# Optional: Display one processed example
print("\nSample tokenized example from the training set:")
print(dataset_samsum_pt["train"][0])

Loading tokenizer: google/pegasus-cnn_dailymail...


config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Loaded dataset successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})
Tokenizing dataset splits...


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Tokenization completed successfully!
Available splits: ['train', 'test', 'validation']

Sample tokenized example from the training set:
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.', 'input_ids': [12195, 151, 125, 7091, 3659, 107, 842, 119, 245, 181, 152, 10508, 151, 7435, 147, 12195, 151, 125, 131, 267, 650, 119, 3469, 29344, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [12195, 7091, 3659, 111, 138, 650, 10508, 181, 3469, 107, 1]}


## 4. setup modle adn huggingface trainer

we load teh base weights of teh pegasus model, move it to teh gpu (cuda), adn initialize teh seq2seq trainer configuration.

In [ ]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA is available: {torch.cuda.is_available()}")
print(f"Using training device: {device.upper()}")


model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)
model_pegasus.to(device)


seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_pegasus
)


training_args = TrainingArguments(
    output_dir="pegasus_samsum_checkpoints",
    num_train_epochs=3,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000000,
    fp16=True,
    report_to="none"
)


trainer = Trainer(
    model=model_pegasus,
    args=training_args,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"],
)

print("Trainer initialized successfully!")

CUDA is available: True
Using training device: CUDA


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Trainer initialized successfully!


## 5. execute fine-tuning

run teh trainig loop. this will take around 30 to 45 minutes on a free colab gpu instance.

In [ ]:
aprint("Starting Pegasus model fine-tuning...")
trainer.train()
print("Fine-tuning run complete!")

Starting Pegasus model fine-tuning...


Step,Training Loss,Validation Loss
500,26.912754,1.499222
1000,23.869417,1.418844
1500,23.337131,1.393545
2000,22.280081,1.379648
2500,22.036677,1.370861
2763,21.638296,1.370154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning run complete!


## 6. export fine-tuned model weights & toknizer

we save teh fine-tuned modle and tokenizer to disk using standard hf formats, then compress them to `.zip` files.

In [ ]:
import os
import shutil

model_export_dir = "pegasus-samsum-model"
tokenizer_export_dir = "tokenizer"

print("Saving model weights...")
os.makedirs(model_export_dir, exist_ok=True)
model_pegasus.save_pretrained(model_export_dir)

print("Saving tokenizer...")
os.makedirs(tokenizer_export_dir, exist_ok=True)
tokenizer.save_pretrained(tokenizer_export_dir)

print("Creating ZIP archives...")
shutil.make_archive(model_export_dir, "zip", model_export_dir)
shutil.make_archive(tokenizer_export_dir, "zip", tokenizer_export_dir)

print("Export completed successfully!")
print("Created:")
print("- pegasus-samsum-model.zip")
print("- tokenizer.zip")

Saving model weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving tokenizer...
Creating ZIP archives...
Export completed successfully!
Created:
- pegasus-samsum-model.zip
- tokenizer.zip


## 7. download files to local machine

choose **option a** to download files directly through teh browser, or **option b** to save them to your gogle drive.

###  save directly to google drive (recommended & highly reliable)

In [ ]:
from google.colab import drive
import os
import shutil

try:
    print("Mounting Google Drive...")
    drive.mount("/content/drive")

    drive_dest = "/content/drive/MyDrive/pegasus_samsum_exports"
    os.makedirs(drive_dest, exist_ok=True)

    print(f"Copying ZIP files to: {drive_dest}")

    shutil.copy(
        "pegasus-samsum-model.zip",
        os.path.join(drive_dest, "pegasus-samsum-model.zip")
    )

    shutil.copy(
        "tokenizer.zip",
        os.path.join(drive_dest, "tokenizer.zip")
    )

    print("ZIP files successfully saved to Google Drive!")

except Exception as e:
    print(f"Google Drive operation failed: {e}")

Mounting Google Drive...
Google Drive operation failed: Error: credential propagation was unsuccessful


In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/text-sum"
os.makedirs(drive_path, exist_ok=True)

print("Saving project to Google Drive (text-sum)...")


if os.path.exists("pegasus-samsum-model"):
    shutil.copytree(
        "pegasus-samsum-model",
        os.path.join(drive_path, "model"),
        dirs_exist_ok=True
    )


if os.path.exists("tokenizer"):
    shutil.copytree(
        "tokenizer",
        os.path.join(drive_path, "tokenizer"),
        dirs_exist_ok=True
    )


if os.path.exists("pegasus_samsum_checkpoints"):
    shutil.copytree(
        "pegasus_samsum_checkpoints",
        os.path.join(drive_path, "checkpoints"),
        dirs_exist_ok=True
    )


if os.path.exists("samsum_dataset"):
    shutil.copytree(
        "samsum_dataset",
        os.path.join(drive_path, "dataset"),
        dirs_exist_ok=True
    )



Mounted at /content/drive
Saving project to Google Drive (text-sum)...
DONE ✅ Everything saved to: MyDrive/text-sum
